# Chapter 51. Data Activation: Reverse ETL, APIs & System Integration

*The runnable half of this chapter.* Every code cell below is the chapter's own code, and the words around it are the chapter's own explanation. Run the cells in order: each one uses names made by the cells before it.

Riverstone Supplies is the single worked example throughout the book, so the data here is the same data you meet in every other chapter.

Built from `manuscript/ch51-data-activation.md`.

# Chapter 51. Data Activation: Reverse ETL, APIs & System Integration

*Part 5 — Data Engineering, Integration & Scale*

> **Chapter at a glance**
>
> **You will learn to:** explain why a number sitting in a warehouse or a dashboard changes nothing on its own · describe reverse ETL and the kinds of fields companies sync back into operational systems · write to a system through an API with authentication, upserts, pagination, rate limits, and retries · make a write **idempotent** with an idempotency key, and prove a retried write doesn't double up · choose a system of record and a conflict rule before two systems can disagree · trigger a webhook, receive the event within seconds instead of polling, and check its signature · compare integration patterns: point-to-point, hub-and-spoke, message queues, and iPaaS · decide between custom code, orchestrator tasks, low-code tools, and enterprise platforms · explain why RPA is fragile, and when file-based integration (SFTP, EDI) is still the right answer · build an audit trail and a reconciliation report for writes, not only reads.
>
> **Before you start:** Chapter 45 (ingestion, idempotency, APIs), Chapter 46 (orchestration and delivery), Chapter 47 (contracts and incidents), and Chapters 12 and 13 (SQL, including window functions).
>
> **Time needed:** 14–18 hours of reading and practice, spread over two to three weeks.
>
> **Tools:** Python with `psycopg2` and `requests` (both already installed, in Chapters 45 and 18). PostgreSQL 16. The Chapter 51 companion folder, which includes a sandbox CRM and a webhook receiver, both running on your own computer.
>
> **Practice data:** Riverstone's leads and their stage history, and Riverstone's customers and orders, from `riverstone_source`, a fresh copy of `riverstone_2025`. Every output shown is from a real run against the sandbox CRM.

---

## Why this matters

Every chapter in Part 5 so far has moved data **toward** the warehouse: ingest it, clean it, model it, check it, store it well. That data now knows things nobody in the business has noticed yet: which leads are hot, which customers have quietly stopped ordering, which machine is drifting hot. None of that changes anything by sitting in a table.

**Data activation** is the last mile: pushing what the warehouse knows back into the systems where people actually work. A sales rep doesn't open a dashboard before every call; they open the CRM. A dispatch clerk doesn't run a query; they look at the ERP. If the insight isn't there, it doesn't exist for them.

This is also where the stakes change. A wrong number in a dashboard is embarrassing. A wrong number **written into the CRM** can change what a customer is charged, what a rep is told to say, or which orders get held. Chapter 47's severities apply here at their sharpest: a bad reverse-ETL sync is not a late report, it's data corruption in a system other people depend on. This chapter builds that sync properly, on Riverstone's own leads and orders, against a sandbox CRM that behaves like a real one: it needs a token, it fails sometimes, and it remembers what you've already told it.

---

## In plain English

Think about a factory floor with a whiteboard and a computer system.

- The computer system **knows** which machines are running hot, from sensors nobody on the floor watches directly.
- Someone has to **write it on the whiteboard** the supervisor actually reads. That's activation.
- If the same warning gets written twice because someone's marker slipped, the supervisor might stop a machine that's actually fine. **Writing it exactly once, however many times you try,** is idempotency.
- The whiteboard and the computer system can **disagree**: if a supervisor wipes off a warning early and the system writes it back, whose word counts? That's the **system of record**.
- A runner who **rushes over the moment something changes**, instead of waiting for someone to walk past the board, is a **webhook**.
- If the computer system has no way to talk to the whiteboard at all, someone might have to **transcribe it by hand every hour**. That's what RPA and file drops stand in for, and why they're fragile: a hand copying a whiteboard makes mistakes a direct wire never would.

---

## 51.0 Setting up the practice environment

This chapter writes into another system, so it needs something safe to write into. The companion folder `companion/ch51/` has everything, and all of it runs on your own computer:

| File | What it does |
|---|---|
| `reset_ch51.py` | Creates `riverstone_source`, a fresh copy of `riverstone_2025` (as in Chapter 45), and empties the sandbox's storage. Run it any time to start again. |
| `crm_sandbox.py` | The **sandbox CRM**: a small local web server that behaves like a real CRM's API, for reading and for writing. It holds Riverstone's leads and customer accounts. |
| `webhook_receiver.py` | A small local server that receives webhooks (section 51.6). |
| `lead_scores.sql`, `follow_up_due.sql` | The two queries you'll read in section 51.2. |
| `sync_leads.py` | The chapter's computations and keys collected in one file, for the project. |

### Step 1. Your practice folder

As in Chapter 45, copy the folder `companion/ch51` and paste it inside `work`, so you have `work/ch51` to practise in.

### Step 2. Nothing new to install

`psycopg2-binary` was installed in Chapter 45 (section 45.2), and `requests` and `python-dotenv` in Chapter 18. The other modules this chapter uses (`json`, `time`, `hashlib`, and `hmac`) come with Python.

### Step 3. Settings in `.env`

Copy the `.env` file from `work/ch45` into `work/ch51`, so `RIVERSTONE_SOURCE` points at the same database, and add two lines:

**The chapter's output:**

```
CRM_API_TOKEN=practice-token-51
WEBHOOK_SECRET=practice-secret-51
```

- **`CRM_API_TOKEN`** is the token the CRM gave your integration, to prove who is calling (Chapter 45, section 45.9).
- **`WEBHOOK_SECRET`** is a secret shared between the CRM and your webhook receiver, used to check that a webhook really came from the CRM (section 51.6).

Both values here are the sandbox's public practice values. Real ones are secrets: they live in `.env` or a secrets store, never in code or in Git. Chapter 52 (section 52.6) shows every layer of that in production.

`reset()` connects to PostgreSQL's maintenance database, `postgres`, to copy the database. If your PostgreSQL needs a password (usual on Windows), add a third line with your own password, in the same form as Chapter 45's connection string: `RIVERSTONE_PG=dbname=postgres user=postgres password=your-password host=localhost`.

### Step 4. Start the notebook

In VS Code, create a notebook `ch51.ipynb` in `work/ch51`, with the `.venv` kernel (Chapter 17, section 17.0). Run each code block in this chapter as its own cell, top to bottom, **without restarting the kernel**: later cells use names that earlier cells create, such as `scores` and `sync_record`. If you restart, run the cells again from the top.

The first cell:

In [ ]:
import os, json, time, hashlib, hmac
import psycopg2, requests
from dotenv import load_dotenv

load_dotenv()
from reset_ch51 import reset
from crm_sandbox import seed_crm, simulate_failure, start_server as start_crm
from webhook_receiver import start_server as start_receiver, read_inbox

reset()
SRC = os.environ.get("RIVERSTONE_SOURCE", "dbname=riverstone_source")
print("riverstone_source ready")

> **Not run here.** This cell needs something this machine does not have: a database, an API key, a running service, or command-line arguments. Run it on your own machine, with the setup the chapter describes, and you should see what the chapter shows.

**What the chapter shows:**

```
riverstone_source ready
```

**How it works, line by line.**

- `json` turns Python objects into JSON text (Chapter 17), `time` waits and measures time, `hashlib` computes fingerprints (Chapter 45, section 45.5), and `hmac` signs messages with a secret (section 51.6).
- `load_dotenv()` copies `.env` into the environment. It comes before the companion imports, because they read their settings when they load.
- `from crm_sandbox import ... start_server as start_crm` imports a function under a new name. Both companion servers have a function called `start_server`; `as` gives each its own name, so the second import doesn't replace the first.
- `reset()` rebuilds `riverstone_source` from `riverstone_2025` and empties the sandbox's storage folders, so every run of this chapter starts from the same state.
- `SRC` is the connection string, exactly as in Chapter 45.

Now start the two servers and fill the sandbox CRM:

In [ ]:
crm = start_crm()
receiver = start_receiver()
leads_n, accounts_n = seed_crm()
BASE = "http://127.0.0.1:8051/api"
TOKEN = os.environ.get("CRM_API_TOKEN", "practice-token-51")   # practice default only
HEADERS = {"Authorization": f"Bearer {TOKEN}"}
print("CRM:", crm.server_address, " receiver:", receiver.server_address)
print("the CRM holds", leads_n, "leads and", accounts_n, "customer accounts")

> **Not run here.** This cell depends on a cell above that could not run here, so its names were never created. Run it on your own machine, with the setup the chapter describes, and you should see what the chapter shows.

**What the chapter shows:**

```
CRM: ('127.0.0.1', 8051)  receiver: ('127.0.0.1', 8052)
the CRM holds 43 leads and 23 customer accounts
```

- `start_crm()` and `start_receiver()` start the two servers in the background of the notebook, at ports 8051 and 8052 on this computer, as Chapter 45's practice API did at 8045.
- `seed_crm()` fills the sandbox from `riverstone_source`: all of Riverstone's leads, and an account for each customer except the newest, customer 24, who signed up in November and has no CRM account yet (section 51.4 creates it). It returns the two counts.
- `BASE` is the start of every address the CRM answers on. `TOKEN` comes from the environment; the default after it is the sandbox's public token, so the notebook runs even without `.env`. `HEADERS` sends it as a bearer token.

> **Tool note.** The outputs in this chapter were produced with Python 3.11.15, requests 2.33.1, psycopg2 2.9.13, python-dotenv 1.2.3, and PostgreSQL 16.13.

---

## 51.1 The loop nobody finishes

Chapter 7 drew the full flow from a question to an answer. Here's the piece most companies build, and then stop before finishing.

![A left to right flow of four boxes: source systems (ERP, CRM, marketing tools), ingestion (Chapter 45), the warehouse, and data products (reports, emails, dashboards). A label above data products says most companies stop here. A purple arrow labelled this chapter leaves the top of the warehouse and runs back into the source systems, closing the loop into the systems people work in.](figures/fig51-1-the-full-loop.svg)

*Figure 51.1 — Most data platforms stop at the dashboard. Activation closes the loop back into the systems people work in every day.*

A **dashboard** is pull: someone has to open it, notice something, and act. A **synced field in the CRM** is push: it's sitting where the action already happens, next to the customer's name, when the rep is already looking. The difference in how often it gets used is not small.

**Fields worth syncing back**, and what they change:

| Field | Computed from | Changes what |
|---|---|---|
| **Lead score** | Stage, source, recency (this chapter) | Which leads a rep calls first |
| **Follow-up-due flag** | Order dates and delivery status (this chapter) | Which quiet customers a rep calls this week |
| **Customer health score** | Order frequency, support tickets, payment history | Who account management checks in on |
| **Credit limit or hold** | Payment history, exposure | Whether an order can be placed at all |
| **Reorder flag** | Stock levels, usage rate | Whether a purchase order gets raised |
| **Overdue-payment flag** | Invoices and payments | Whether finance chases a customer this week |
| **Churn risk** | Usage decline, contract dates | Whether a renewal call gets scheduled |

Every one of these is a number the warehouse can compute, once it has the data. The work in this chapter is getting it to land somewhere it changes a decision, safely.

---

## 51.2 Computing what to sync

Riverstone will sync two things into its CRM: a **lead score** on each lead, so reps call the right leads first, and a **follow-up-due flag** on each customer account, so reps call the customers who have gone quiet.

**"Today" in this chapter.** Throughout this chapter, today is **6 January 2026**, the same simulated calendar as Chapter 46: the nightly run starts at 06:30 IST that morning. Every date rule below counts back from that day, which the SQL writes as `DATE '2026-01-06'`, so every run of the chapter gives the same answer.

### Lead scoring

The rule is deliberately simple and fully documented, because a score nobody can explain doesn't earn trust from the sales team:

> **score = points for the lead's current stage + points for how it arrived + a recency bonus if it entered that stage in the last 14 days**, capped at 100. A Lost lead scores 0, whatever else is true.

| Current stage | Points |
|---|---|
| New | 10 |
| Contacted | 30 |
| Quoted | 55 |
| Won | 100 |
| Lost | 0, and the whole score is 0 |

| Source | Points |
|---|---|
| Trade fair | 20 |
| Referral | 15 |
| Website | 10 |
| IndiaMART listing | 10 |
| Cold call | 5 |

- **Recency:** +10 if the lead entered its current stage on or after 23 December 2025, 14 days before today.
- **Lost:** a lost lead should never rank above an active one, so it scores 0 whatever its source or recency.
- **The cap:** only a Won lead can pass 100 (100 plus its source points), so every Won lead scores exactly 100. The highest score for a lead still in play is Quoted 55 + Trade fair 20 + recency 10 = 85.

**By hand first.** To score a lead you need its source and its current stage, with the date it entered that stage. The table `lead_stage_history` has one row for each stage a lead has entered. This query, run in psql or DBeaver against `riverstone_source` (Chapter 12), picks each lead's newest row:

<!-- db: riverstone_source -->

**`sql`, not run by this notebook.** Run it in the tool the chapter names.

```sql
WITH latest AS (
    SELECT lead_id, stage, entered_at,
           ROW_NUMBER() OVER (PARTITION BY lead_id ORDER BY entered_at DESC) AS rn
    FROM lead_stage_history
)
SELECT l.lead_id, l.company_name, l.source, s.stage, s.entered_at::date AS stage_entered
FROM leads AS l
JOIN latest AS s ON s.lead_id = l.lead_id AND s.rn = 1
WHERE l.lead_id <= 7
ORDER BY l.lead_id;
```

**The chapter's output:**

```
 lead_id |   company_name    |  source   |   stage   | stage_entered 
---------+-------------------+-----------+-----------+---------------
       1 | Anand Stores      | Referral  | Contacted | 2025-04-13
       2 | Bright Kitchens   | Website   | Quoted    | 2025-06-29
       3 | Bright Kitchens   | Website   | New       | 2025-06-12
       4 | BRIGHT KITCHENS   | Website   | New       | 2025-06-12
       5 | Cafe Mocha Lane   | Website   | New       | 2025-01-02
       6 | Delta Hospitality | Cold call | Lost      | 2025-05-27
       7 | Elite Mart        | Referral  | Won       | 2025-07-23
(7 rows)
```

- `ROW_NUMBER() OVER (PARTITION BY lead_id ORDER BY entered_at DESC)` numbers each lead's stages from the newest (Chapter 13), so `rn = 1` is the stage the lead is in now.
- `::date` keeps only the date part of `entered_at` (Chapter 12's cast shorthand, section 12.8).
- `WHERE l.lead_id <= 7` keeps the printout short; the real query scores every lead.

Three of these, by hand:

| Lead | Stage | Source | Recency | Score |
|---|---|---|---|---|
| 2, Bright Kitchens | Quoted 55 | Website 10 | 0 (entered 29 June 2025) | 55 + 10 + 0 = **65** |
| 6, Delta Hospitality | Lost | (Cold call 5) | 0 | Lost, so **0** |
| 7, Elite Mart | Won 100 | Referral 15 | 0 | 115, capped to **100** |

No lead in this data entered its stage on or after 23 December (the latest date is 4 December), so nobody gets the recency bonus today. Exercise 4 has a lead that does.

**The same rule in SQL.** The file `lead_scores.sql` in your practice folder turns the two tables and the three rules into SQL. The next Python cell runs it.

**`sql`, not run by this notebook.** Run it in the tool the chapter names.

```sql
WITH latest AS (
    SELECT lead_id, stage, entered_at,
           ROW_NUMBER() OVER (PARTITION BY lead_id ORDER BY entered_at DESC) AS rn
    FROM lead_stage_history
),
points AS (
    SELECT l.lead_id, l.company_name, l.source, s.stage,
           CASE s.stage WHEN 'New' THEN 10 WHEN 'Contacted' THEN 30
                        WHEN 'Quoted' THEN 55 WHEN 'Won' THEN 100 ELSE 0 END AS stage_points,
           CASE l.source WHEN 'Trade fair' THEN 20 WHEN 'Referral' THEN 15
                         WHEN 'Website' THEN 10 WHEN 'IndiaMART listing' THEN 10
                         WHEN 'Cold call' THEN 5 ELSE 0 END AS source_points,
           CASE WHEN s.entered_at >= DATE '2026-01-06' - 14 THEN 10 ELSE 0 END AS recency_points
    FROM leads AS l
    JOIN latest AS s ON s.lead_id = l.lead_id AND s.rn = 1
)
SELECT lead_id, company_name, source, stage,
       CASE WHEN stage = 'Lost' THEN 0
            ELSE LEAST(stage_points + source_points + recency_points, 100) END AS score
FROM points
ORDER BY lead_id;
```

- `latest` is the query you just ran, without the `WHERE`.
- `points` works out the three parts, one column each, with `CASE` (Chapter 12). **`CASE s.stage WHEN 'New' THEN 10 ...`** is the short form of `CASE WHEN s.stage = 'New' THEN 10 ...`: it compares one column with each value in turn. A stage or source not in the table gets `ELSE 0`.
- **`DATE '2026-01-06' - 14`** subtracts 14 days from a date: 23 December 2025, the start of the recency window.
- The last `SELECT` applies the two overrides. **`LEAST(a, b)`** returns the smaller of its values, so `LEAST(total, 100)` is the cap. The outer `CASE` makes a Lost lead 0 before anything else.

Now run it from Python and keep the scores:

In [ ]:
def fetch_rows(sql, params=None):
    conn = psycopg2.connect(SRC)
    with conn, conn.cursor() as cur:
        cur.execute(sql, params)
        rows = cur.fetchall()
    conn.close()
    return rows

with open("lead_scores.sql") as f:
    score_rows = fetch_rows(f.read())
scores = {str(lead_id): {"company_name": name, "stage": stage, "score": score}
          for lead_id, name, source, stage, score in score_rows}
print(len(scores), "leads scored")
for lead_id in ["2", "6", "7"]:
    print(lead_id, scores[lead_id])

> **Not run here.** This cell depends on a cell above that could not run here, so its names were never created. Run it on your own machine, with the setup the chapter describes, and you should see what the chapter shows.

**What the chapter shows:**

```
43 leads scored
2 {'company_name': 'Bright Kitchens', 'stage': 'Quoted', 'score': 65}
6 {'company_name': 'Delta Hospitality', 'stage': 'Lost', 'score': 0}
7 {'company_name': 'Elite Mart', 'stage': 'Won', 'score': 100}
```

- `fetch_rows()` is a smaller version of Chapter 45's helper: it opens a connection, runs one query, and returns the rows as a list of tuples. Its second argument, `params`, would fill `%s` placeholders in the query with values, as in Chapter 45; this chapter's queries have none, so it stays `None`.
- `with open("lead_scores.sql") as f:` opens the file, and `f.read()` reads it as one string: the query.
- The dictionary comprehension (Chapter 17) makes one entry per row. `for lead_id, name, source, stage, score in score_rows` unpacks each row's five columns into five names. The key is `str(lead_id)`, text, because the CRM's addresses and JSON use text ids.

**Reading it.** All 43 leads are scored, and the three agree with the table worked by hand: 65, 0, and 100. Check a computed field by hand like this every time: a score nobody can rebuild from its own rule is a score nobody should trust, however tidy the code looks.

> **Simplification note.** A learned lead-scoring model would use more signals, such as the number of contacts, the deal size, and what happened to similar past leads. That's a classification model, the kind Part 4 built on Riverstone's CRM leads (Chapters 35 to 39); its predicted probability could be synced with exactly this chapter's mechanics. This score is deliberately transparent: every point traces to a rule a sales manager can read in one sentence, which matters more early on than a cleverer model nobody can explain.

### The follow-up-due flag

Riverstone's finance system tracks invoices and payments separately, and this book doesn't have that data, so the chapter doesn't pretend to compute an overdue-payment flag. It syncs a flag it can compute honestly from orders:

> A customer is **follow-up due** if they ordered in the last six months (since 6 July 2025) but have had **no delivered order in the last 45 days** (since 22 November 2025).

This is a recency signal, a customer who has gone quiet, not a payment status. A customer who owes money but took a delivery last week isn't flagged. Any real deployment should replace or sharpen it with its owner's actual definition, agreed as a Chapter 47 data contract. The query is in `follow_up_due.sql`:

**`sql`, not run by this notebook.** Run it in the tool the chapter names.

```sql
SELECT c.customer_id, c.customer_name,
       MAX(o.order_date) AS last_order,
       MAX(CASE WHEN o.status = 'Delivered' THEN o.order_date END) AS last_delivered
FROM customers AS c
JOIN orders AS o ON o.customer_id = c.customer_id
GROUP BY c.customer_id, c.customer_name
HAVING MAX(o.order_date) >= DATE '2026-01-06' - INTERVAL '6 months'
   AND COALESCE(MAX(CASE WHEN o.status = 'Delivered' THEN o.order_date END), DATE '1900-01-01')
       < DATE '2026-01-06' - 45
ORDER BY c.customer_id;
```

**The chapter's output:**

```
 customer_id |   customer_name   | last_order | last_delivered 
-------------+-------------------+------------+----------------
           2 | Patel Kitchenware | 2025-11-09 | 2025-11-09
           8 | Blue Bay Cafe     | 2025-11-09 | 2025-11-09
           9 | Lakeview Resorts  | 2025-12-21 | 2025-10-15
          13 | Om Sai Provisions | 2025-07-22 | 2025-07-22
          20 | Tasty Tiffins     | 2025-10-26 | 2025-10-26
          22 | Festive Gifts Co  | 2025-11-19 | 2025-11-19
(6 rows)
```

- `MAX(CASE WHEN o.status = 'Delivered' THEN o.order_date END)` is the date of the latest delivered order. A `CASE` with no `ELSE` gives `NULL` for other orders, and `MAX` ignores `NULL`s.
- `HAVING` keeps the groups, one per customer, that pass both tests (Chapter 12). `INTERVAL '6 months'` is Chapter 13's date arithmetic.
- `COALESCE(..., DATE '1900-01-01')` turns "never had a delivery" into a very old date, so such a customer is flagged too, instead of silently dropping out as `NULL`.

**Reading it.** Six customers are follow-up due. Look at Lakeview Resorts: it placed an order on 21 December, but that order is still Pending, and its last delivery was 15 October, so it's flagged. Should an order on its way count? The rule says no. Whether that's right is a question for the flag's owner, which is exactly why the definition belongs in a contract, not only in code.

The sync needs a value for **every** customer, `True` or `False`, so that a customer who orders again gets the flag cleared:

In [ ]:
with open("follow_up_due.sql") as f:
    due = {str(row[0]) for row in fetch_rows(f.read())}
customer_ids = [str(row[0]) for row in fetch_rows("SELECT customer_id FROM customers ORDER BY customer_id")]
follow_up = {cid: cid in due for cid in customer_ids}
print(len(follow_up), "customers,", sum(follow_up.values()), "follow-up due:", sorted(due, key=int))

> **Not run here.** This cell depends on a cell above that could not run here, so its names were never created. Run it on your own machine, with the setup the chapter describes, and you should see what the chapter shows.

**What the chapter shows:**

```
24 customers, 6 follow-up due: ['2', '8', '9', '13', '20', '22']
```

- `{str(row[0]) for row in ...}` is a **set** comprehension: the shape of Chapter 17's dictionary comprehension, but with no `key: value`, so it builds a set (Chapter 17), where each id appears once. `row[0]` is the first column, the customer id.
- `cid in due` is `True` or `False`, so `follow_up` maps every customer to its flag.
- `sum()` of booleans counts the `True`s, because `True` counts as 1. `sorted(due, key=int)` sorts the ids as numbers, so "13" comes after "9".

---

## 51.3 Writing to a system through an API

Chapter 45 read from a CRM's API. Writing is the same shape with sharper edges: a bad read wastes a query; a bad write changes someone else's system.

### Authentication for writes

Most write APIs use the same authentication as reads (Chapter 45's bearer tokens), but with a tighter **scope** (the list of things a token is allowed to do): a read scope might cover the whole CRM, while a write scope might be limited to a few fields or objects. **OAuth 2.0** is the standard for delegated write access: your application gets a short-lived **access token** after a user, or a **service account** (a login that belongs to an application rather than a person), grants permission, and it refreshes the token before it expires, rather than holding one permanent password-like key forever. The sandbox CRM uses a fixed bearer token to keep the examples short; production integrations almost always use OAuth for anything that writes.

### Four HTTP methods for writing

Chapter 2 introduced requests, Chapter 18 sent `GET` requests, and Chapter 20 sent a `POST` to a chat channel's webhook. Writing uses four methods, and the difference between two of them causes real damage:

| Method | What it asks the system to do |
|---|---|
| `GET` | Read a record. Changes nothing. |
| `POST` | Create a new record. |
| `PUT` | **Replace** the whole record with what you send. Any field you leave out is gone. |
| `PATCH` | **Change only** the fields you send. Everything else stays as it was. |

See the difference on lead 1. First, read it:

In [ ]:
lead = requests.get(f"{BASE}/leads/1", headers=HEADERS, timeout=10).json()
print(lead)

> **Not run here.** This cell depends on a cell above that could not run here, so its names were never created. Run it on your own machine, with the setup the chapter describes, and you should see what the chapter shows.

**What the chapter shows:**

```
{'lead_id': '1', 'company_name': 'Anand Stores', 'email': 'anandstores@example.com', 'source': 'Referral', 'owner_id': 3}
```

Now send only a score, with `PUT`, and read the lead again:

In [ ]:
resp = requests.put(f"{BASE}/leads/1", json={"score": 45}, headers=HEADERS, timeout=10)
print(resp.status_code, resp.json())
print(requests.get(f"{BASE}/leads/1", headers=HEADERS, timeout=10).json())

> **Not run here.** This cell depends on a cell above that could not run here, so its names were never created. Run it on your own machine, with the setup the chapter describes, and you should see what the chapter shows.

**What the chapter shows:**

```
200 {'record': 'leads/1', 'action': 'replaced'}
{'lead_id': '1', 'score': 45}
```

- `requests.put(url, json=...)` sends a `PUT` with a JSON body. `json=` turns the dictionary into JSON text and tells the server so.

**Reading it.** The CRM did exactly what `PUT` means: the record is now the id plus a score. The company name, the email, the source, and the owner are gone. Put the sandbox back, and try the same change with `PATCH`:

In [ ]:
seed_crm()                                  # put the sandbox back as it was
resp = requests.patch(f"{BASE}/leads/1", json={"score": 45}, headers=HEADERS, timeout=10)
print(resp.status_code, resp.json())
print(requests.get(f"{BASE}/leads/1", headers=HEADERS, timeout=10).json())

> **Not run here.** This cell depends on a cell above that could not run here, so its names were never created. Run it on your own machine, with the setup the chapter describes, and you should see what the chapter shows.

**What the chapter shows:**

```
200 {'record': 'leads/1', 'action': 'updated', 'fields': ['score'], 'replayed': False}
{'lead_id': '1', 'company_name': 'Anand Stores', 'email': 'anandstores@example.com', 'source': 'Referral', 'owner_id': 3, 'score': 45}
```

**Reading it.** `PATCH` changed one field, `score`, and left everything else alone. A reverse-ETL sync should almost always use `PATCH`, and name only the fields it owns. This chapter's closing story is what happens when it doesn't.

### The idempotency key

A sync will sometimes send the same write twice: a retry after a timeout, or a whole run started again. The protection is an **idempotency key**: a short text sent with each write, in a header called `Idempotency-Key`. The CRM remembers every key it has applied. If a write arrives with a key it has already seen, it changes nothing and sends back its earlier answer, marked `"replayed": True`.

So the key must be **the same whenever the write is the same**. Build it from the record and the values being written:

In [8]:
def idempotency_key(record, payload):
    content = record + ":" + json.dumps(payload, sort_keys=True)
    return hashlib.sha256(content.encode()).hexdigest()[:16]

print(idempotency_key("leads/2", {"score": 65, "stage": "Quoted"}))
print(idempotency_key("leads/2", {"stage": "Quoted", "score": 65}))
print(idempotency_key("leads/2", {"score": 66, "stage": "Quoted"}))

d6f8907536e7b40d
d6f8907536e7b40d
50bbc6a7d7d61904


- `record` is the record's address in the CRM, such as `"leads/2"`, so two records with the same values get different keys.
- **`json.dumps(payload, sort_keys=True)`** turns the values into text with the keys in alphabetical order. The same dictionary always gives the same text, whatever order it was built in: that's why the first two keys match.
- `hashlib.sha256(...).hexdigest()` is the SHA-256 fingerprint from Chapter 45 (section 45.5); `.encode()` turns the text into bytes first, as there. `[:16]` keeps the first 16 of its 64 characters: plenty to tell one write from another, and shorter in headers and logs.

**Reading it.** The same values give the same key, in any order; a score of 66 instead of 65 gives a completely different one.

### One write, step by step

Build the first write, for lead 1:

In [ ]:
payload = {"score": scores["1"]["score"], "stage": scores["1"]["stage"]}
RUN_TIME = "2026-01-06T06:30:00+05:30"
body = {**payload, "score_updated_at": RUN_TIME}
key = idempotency_key("leads/1", payload)
print(body)
print({**HEADERS, "Idempotency-Key": key})

> **Not run here.** This cell depends on a cell above that could not run here, so its names were never created. Run it on your own machine, with the setup the chapter describes, and you should see what the chapter shows.

**What the chapter shows:**

```
{'score': 45, 'stage': 'Contacted', 'score_updated_at': '2026-01-06T06:30:00+05:30'}
{'Authorization': 'Bearer practice-token-51', 'Idempotency-Key': 'b5564a8c80e53ded'}
```

- `payload` holds the values the sync owns: the score and the stage it was computed from.
- **`{**payload, "score_updated_at": RUN_TIME}`** builds a new dictionary: `**payload` copies every key and value of `payload` into it, and then one more key is added. `payload` itself is unchanged.
- `score_updated_at` tells a rep in the CRM when the score was last calculated: the run at 06:30 IST on 6 January. The key is built from `payload`, **not** from `body`: the time says when the run happened, not what was written, and a key that includes a time changes every time (section 51.4 comes back to this).
- The last line shows the headers the write will carry: the token, plus the key.

The function that sends one write, and retries only what might succeed next time:

In [ ]:
TIMEOUT = 2                     # seconds; the sandbox answers in a few milliseconds
RETRY_STATUSES = {429, 503}

def sync_record(record, body, key, max_attempts=3):
    headers = {**HEADERS, "Idempotency-Key": key}
    for attempt in range(1, max_attempts + 1):
        try:
            resp = requests.patch(f"{BASE}/{record}", json=body, headers=headers, timeout=TIMEOUT)
        except (requests.Timeout, requests.ConnectionError) as e:
            problem, wait = type(e).__name__, 2 ** (attempt - 1)
        else:
            if resp.status_code not in RETRY_STATUSES:
                resp.raise_for_status()
                return resp.json()
            problem = f"{resp.status_code} {resp.reason}"
            wait = int(resp.headers.get("Retry-After", 2 ** (attempt - 1)))
        if attempt < max_attempts:
            print(f"  {record}: {problem}, waiting {wait}s (attempt {attempt})")
            time.sleep(wait)
    raise RuntimeError(f"{record}: gave up after {max_attempts} attempts (last: {problem})")

print(sync_record("leads/1", body, key))

> **Not run here.** This cell depends on a cell above that could not run here, so its names were never created. Run it on your own machine, with the setup the chapter describes, and you should see what the chapter shows.

**What the chapter shows:**

```
{'record': 'leads/1', 'action': 'updated', 'fields': ['score', 'score_updated_at', 'stage'], 'replayed': False}
```

**How it works, line by line.** The loop is Chapter 45's `get_with_retry()` (section 45.9), turned into a write:

- `requests.patch(f"{BASE}/{record}", json=body, ...)` sends a `PATCH` to the record's address, such as `.../api/leads/1`. A part of the address that names one record is called a **path parameter**.
- The **key arrives from outside** the function, and the headers are built once, before the loop, so every attempt sends the same key. That's the whole point: a retry must look like the same write.
- **Timeouts and dropped connections** (`requests.Timeout`, `requests.ConnectionError`) are retried, with waits of 1, then 2 seconds. `TIMEOUT` is short because the sandbox is on your own computer; Chapter 45 used 10 seconds for a real API.
- **429 Too Many Requests** and **503 Service Unavailable** are retried, waiting for `Retry-After` seconds when the CRM sends that header. Any other error, such as 401 or 404, goes to `raise_for_status()` and stops at once: retrying won't fix it.
- The message is printed only when another attempt follows (`attempt < max_attempts`); after the last attempt the function raises a clear error instead.
- The answer, `resp.json()`, says which record changed, whether it was updated or created, which fields, and `"replayed": False`: this write was new.

Now the first nightly sync of every lead. The sandbox has one more piece of **test scaffolding**, `simulate_failure()`, which makes the CRM fail one write on purpose, so you can watch the retry work. A real CRM doesn't have it.

In [ ]:
simulate_failure("leads/4", "429")      # the CRM will refuse lead 4's next write once
outcomes = {"new": 0, "replayed": 0}
for lead_id, s in scores.items():
    payload = {"score": s["score"], "stage": s["stage"]}
    key = idempotency_key(f"leads/{lead_id}", payload)
    result = sync_record(f"leads/{lead_id}", {**payload, "score_updated_at": RUN_TIME}, key)
    if result["replayed"]:
        outcomes["replayed"] += 1
    else:
        outcomes["new"] += 1
print(outcomes)

> **Not run here.** This cell depends on a cell above that could not run here, so its names were never created. Run it on your own machine, with the setup the chapter describes, and you should see what the chapter shows.

**What the chapter shows:**

```
  leads/4: 429 Too Many Requests, waiting 1s (attempt 1)
{'new': 42, 'replayed': 1}
```

- The loop goes through every scored lead, builds its payload and key, and sends it.
- `outcomes` counts how many writes were new and how many the CRM recognized as repeats.

**Reading it.** Lead 4's first attempt was refused with 429; the loop waited the one second the CRM asked for and tried again with the same key, and it went through. 42 writes were new. One was a replay: lead 1, whose identical write the previous cell had already sent. The CRM recognized the key and changed nothing.

---

## 51.4 Making writes idempotent, and proving it

Say it as plainly as Chapter 45 did for loads: **assume a write can happen more than once, and design for that on purpose.** A network timeout doesn't tell you whether the request arrived; retrying is correct, and correctness depends on the retry being harmless.

### Upserts: creating a record that isn't there yet

The follow-up flags go on customer **accounts**. Customer 24, Prime Wholesale, has no CRM account yet, so an ordinary update would fail with 404. The sandbox's accounts address works as an **upsert**: a `PATCH` to `accounts/<ERP customer id>` updates the account with that id, or creates it if there isn't one. It's Chapter 45's `INSERT ... ON CONFLICT ... DO UPDATE` (section 45.5), done through an API. Many CRM APIs offer the same thing, often called an upsert by **external id**: an id that belongs to another system, here the ERP's customer id.

In [ ]:
payload = {"follow_up_due": follow_up["24"]}
print(sync_record("accounts/24", payload, idempotency_key("accounts/24", payload)))
print(requests.get(f"{BASE}/accounts/24", headers=HEADERS, timeout=TIMEOUT).json())

> **Not run here.** This cell depends on a cell above that could not run here, so its names were never created. Run it on your own machine, with the setup the chapter describes, and you should see what the chapter shows.

**What the chapter shows:**

```
{'record': 'accounts/24', 'action': 'created', 'fields': ['follow_up_due'], 'replayed': False}
{'customer_id': '24', 'follow_up_due': False}
```

**Reading it.** `"action": "created"`: the account didn't exist, so the upsert made it, with the flag. Sent again, the same write would be a replay, and the account would stay as it is. An upsert is safe to repeat in the same way Chapter 45's SQL upsert was.

### A write that times out

The hardest failure is the one where you don't know what happened. Make the CRM apply the next write to account 9, Lakeview Resorts, and then answer too late:

In [ ]:
simulate_failure("accounts/9", "slow")   # the CRM will apply the next write, then answer 3 s late
payload = {"follow_up_due": follow_up["9"]}
print(sync_record("accounts/9", payload, idempotency_key("accounts/9", payload)))

> **Not run here.** This cell depends on a cell above that could not run here, so its names were never created. Run it on your own machine, with the setup the chapter describes, and you should see what the chapter shows.

**What the chapter shows:**

```
  accounts/9: ReadTimeout, waiting 1s (attempt 1)
{'record': 'accounts/9', 'action': 'updated', 'fields': ['follow_up_due'], 'replayed': True}
```

**Reading it.** After 2 seconds with no answer, `requests` raised `ReadTimeout`, and the function waited a second and tried again. From the notebook's side, the first attempt might have failed on the way there or on the way back; there's no way to tell. In fact it had arrived and been applied. The retry carried the **same key**, so the CRM answered `"replayed": True` and applied nothing twice. For a flag, a double write would be harmless; for "add a ₹500 credit to this account", it would cost money. The key makes both cases safe, and the caller's code doesn't need to know which failure happened. (What if you change `TIMEOUT` to 5 seconds? The CRM's answer, 3 seconds late, would arrive in time, and there'd be no retry at all. A timeout shorter than the other system's slowest normal answer turns slow replies into retries.)

Now sync every account's flag:

In [ ]:
outcomes = {"new": 0, "replayed": 0}
for cid, due_flag in follow_up.items():
    payload = {"follow_up_due": due_flag}
    result = sync_record(f"accounts/{cid}", payload, idempotency_key(f"accounts/{cid}", payload))
    if result["replayed"]:
        outcomes["replayed"] += 1
    else:
        outcomes["new"] += 1
print(outcomes)

> **Not run here.** This cell depends on a cell above that could not run here, so its names were never created. Run it on your own machine, with the setup the chapter describes, and you should see what the chapter shows.

**What the chapter shows:**

```
{'new': 22, 'replayed': 2}
```

**Reading it.** 24 accounts: 22 new writes, and two replays, accounts 24 and 9, which the two cells before had already written.

### A write that's genuinely repeated

Now lead 4. Send its current values again, then a changed value, then the changed value again:

In [ ]:
payload = {"score": scores["4"]["score"], "stage": scores["4"]["stage"]}
print("same values again:", sync_record("leads/4", payload, idempotency_key("leads/4", payload)))

changed = {**payload, "score": payload["score"] + 1}
print("a real change    :", sync_record("leads/4", changed, idempotency_key("leads/4", changed)))
print("that change again:", sync_record("leads/4", changed, idempotency_key("leads/4", changed)))

> **Not run here.** This cell depends on a cell above that could not run here, so its names were never created. Run it on your own machine, with the setup the chapter describes, and you should see what the chapter shows.

**What the chapter shows:**

```
same values again: {'record': 'leads/4', 'action': 'updated', 'fields': ['score', 'score_updated_at', 'stage'], 'replayed': True}
a real change    : {'record': 'leads/4', 'action': 'updated', 'fields': ['score', 'stage'], 'replayed': False}
that change again: {'record': 'leads/4', 'action': 'updated', 'fields': ['score', 'stage'], 'replayed': True}
```

- `changed` copies `payload` and replaces one value: the score plus 1. It stands for a genuine change, such as tomorrow's score after the lead moves stage.

**Reading it.** The first write matches the nightly sync's key exactly, so the CRM replayed it and did nothing. The changed score has a new key, so it was applied (`"replayed": False`). Sending that again was a replay. So far, the key does exactly what it should.

### When a value goes back

Now suppose the score returns to what it was, 20. The warehouse sends 20 again:

In [ ]:
print("back to 20:", sync_record("leads/4", payload, idempotency_key("leads/4", payload)))
print("the CRM now says:", requests.get(f"{BASE}/leads/4", headers=HEADERS, timeout=TIMEOUT).json()["score"])

> **Not run here.** This cell depends on a cell above that could not run here, so its names were never created. Run it on your own machine, with the setup the chapter describes, and you should see what the chapter shows.

**What the chapter shows:**

```
back to 20: {'record': 'leads/4', 'action': 'updated', 'fields': ['score', 'score_updated_at', 'stage'], 'replayed': True}
the CRM now says: 21
```

**Reading it.** This is a real bug. The key for "lead 4, score 20, stage New" was used in the first sync, so the CRM treats the new write as a repeat and ignores it. The warehouse says 20, the CRM says 21, and **every future sync with this key will be ignored too**: a value that goes back to an earlier value can never be sent again. A key built from content alone can't tell "the same write, retried" from "the same values, written again on purpose later".

### The fix: put the run in the key

What should count as "the same write" is: the same record, the same values, **in the same run**. Add the run's id to the key. Here the run id is the day the scheduled run is for, like a Chapter 46 partition key:

In [ ]:
def run_key(record, payload, run_id):
    content = record + ":" + run_id + ":" + json.dumps(payload, sort_keys=True)
    return hashlib.sha256(content.encode()).hexdigest()[:16]

print(run_key("leads/4", payload, "2026-01-06"))
print(run_key("leads/4", payload, "2026-01-06"))
print(run_key("leads/4", payload, "2026-01-07"))

> **Not run here.** This cell depends on a cell above that could not run here, so its names were never created. Run it on your own machine, with the setup the chapter describes, and you should see what the chapter shows.

**What the chapter shows:**

```
a5d6feaec99c52c8
a5d6feaec99c52c8
67329d74f025d291
```

**Reading it.** Within one run, every retry of a write has the same key, so retries are still harmless. Tomorrow's run has a new key, so tomorrow's 20 is applied, even though 20 was sent before. Section 51.7 runs "tomorrow" with this key and checks the result.

Real APIs also **forget** keys after a while. Stripe's API reference, for example, says its keys may be removed once they are at least 24 hours old (checked 29 September 2026). That's one more reason not to depend on a content-only key: whether a revert is ignored would depend on how long the other system remembers.

> **Watch out: an idempotency key must stay the same across every retry of one operation.** Create it once, before the first attempt, and reuse it, as `sync_record()` does. A key built inside the retry loop from "now" or from a fresh random value changes on every attempt and protects nothing. Building it from the record, the values, and the run id guarantees that, and makes it reproducible.

### Choosing what goes in the key

| Built from | Behaviour | Use when |
|---|---|---|
| **Record + values + run id** (recommended, this chapter) | A retry within a run is a no-op; each new run can apply any value, including one sent before | Most reverse-ETL syncs |
| **Record + values only** | A retry is a no-op, but a value that goes back to an earlier one is never sent again (section 51.4) | Only if the receiver forgets keys quickly, and you know how quickly |
| **Record + run id** | One write per record per run, even if the values changed during the run | You want exactly "one write per scheduled run" |
| **A random id (a UUID) created once per operation, and stored with it** | Two identical operations are both applied; a retry of either is not | Payments and order creation, where the same content can legitimately be sent twice (two identical orders). Stripe's API reference suggests this pattern (V4 UUIDs) |

---

## 51.5 System of record and conflict rules

Two systems holding the same fact will eventually disagree. Decide the rule **before** that happens, not while a customer is on the phone.

A **system of record** is the system whose value wins when there's a conflict (Chapter 45, section 45.1). For Riverstone:

| Field | System of record | Why |
|---|---|---|
| Order status, quantities | The ERP | It's where the transaction happens |
| Lead stage, contact history | The CRM | Reps update it directly; the warehouse only reads it |
| Lead score, follow-up flag | **The warehouse**, written into the CRM | They're computed, not entered by a person; the CRM should never let someone hand-edit them and expect it to stick |
| Negotiated discount | **The CRM**, read into the warehouse | A person agreed it; a pipeline must never silently overwrite a human decision |

The last two rows are the trap. A field can be **computed downstream** (safe to overwrite from the warehouse every run) or **entered upstream by a person** (must never be overwritten without an explicit, agreed process). Confusing the two is the single most common cause of the "the sync erased something" incident in this chapter's closing story.

**A minimal conflict policy, written down before the first sync runs:**

1. Name the system of record for every synced field, individually: not "the CRM is the system of record" as a blanket rule.
2. For computed fields, the pipeline always wins; document it as such in the field's description in the CRM itself, so a rep sees "auto-calculated, updates nightly" (and `score_updated_at`) rather than guessing.
3. For fields a person can edit, the sync either **never writes them** or writes only when the field is currently empty, and logs every write for review.
4. Any exception needs a name attached to the decision, and a date.

---

## 51.6 Webhooks: pushing instead of polling

Everything so far runs on a schedule. Some events are worth acting on **the moment they happen**: this chapter's project is a high-value lead that shouldn't sit unnoticed until tomorrow's sync.

A **webhook** (Chapter 2, section 2.8: "an API in reverse") is the receiving side of an event: instead of you asking "anything new?" every minute, the other system calls **you**, with the event, the moment it occurs. Chapter 20 posted to one, a chat channel's incoming webhook; here you receive them. It's the same idea as Chapter 50's log, without the log: a single HTTP push instead of a stream to poll.

First, tell the CRM where to send the event:

In [ ]:
resp = requests.post(f"{BASE}/webhooks", headers=HEADERS, timeout=TIMEOUT,
                     json={"url": "http://127.0.0.1:8052/", "event": "lead.created_high_value"})
print(resp.status_code, resp.json())

> **Not run here.** This cell depends on a cell above that could not run here, so its names were never created. Run it on your own machine, with the setup the chapter describes, and you should see what the chapter shows.

**What the chapter shows:**

```
201 {'registered': {'url': 'http://127.0.0.1:8052/', 'event': 'lead.created_high_value'}}
```

`POST /api/webhooks` **registers** an address and the event that should trigger it. Real webhook providers work the same way: you tell them where to send things, usually in a settings screen or through their API. 201 means "created": the registration exists.

Now a rep creates a high-value lead in the CRM. The sandbox fires `lead.created_high_value` when a new lead's `deal_value` is ₹2,00,000 or more:

In [ ]:
t0 = time.time()
new_lead = requests.post(f"{BASE}/leads", headers=HEADERS, timeout=TIMEOUT, json={
    "company_name": "Grand Horizon Hotels", "email": "procurement@grandhorizon.example",
    "source": "Trade fair", "deal_value": 350000}).json()
print("lead created:", new_lead["lead_id"], new_lead["company_name"])

for _ in range(20):                     # wait up to 2 seconds, checking every 0.1 s
    if read_inbox():
        break
    time.sleep(0.1)
print("webhook arrived in under a second:", time.time() - t0 < 1)
print("event:", read_inbox()[0]["body"])

> **Not run here.** This cell depends on a cell above that could not run here, so its names were never created. Run it on your own machine, with the setup the chapter describes, and you should see what the chapter shows.

**What the chapter shows:**

```
lead created: 9001 Grand Horizon Hotels
webhook arrived in under a second: True
event: {'event_id': 'evt_1', 'event': 'lead.created_high_value', 'lead': {'lead_id': '9001', 'company_name': 'Grand Horizon Hotels', 'email': 'procurement@grandhorizon.example', 'source': 'Trade fair', 'deal_value': 350000}}
```

**How it works, line by line.**

- `time.time()` is the current time in seconds, so `time.time() - t0` is how long everything after `t0` took.
- `POST /api/leads` creates the lead. Its `deal_value` of ₹3,50,000 is over the threshold, so the CRM calls the registered address.
- The `for` loop checks the receiver's inbox up to 20 times, 0.1 seconds apart, and stops as soon as something is there: a small, local version of polling, used only to measure the webhook.
- `read_inbox()[0]["body"]` is the first event received. It carries its own id, `evt_1`, the event's name, and the new lead.

**Reading it.** The event arrived within a second; in fact, it was there before the CRM had even answered the `POST`, because the sandbox calls the webhook **inside** its own request to keep the demo simple. A real provider puts the event in a queue and sends it separately, typically within seconds. Compare that with polling the CRM's leads every five minutes: the same lead could sit unseen for most of that window.

### Making webhook receivers safe

A webhook receiver is a small public API, and it inherits every lesson from Chapters 45 and 46:

- **Verify the sender.** Anyone who finds the address can send it a request. Real providers sign each request (usually an HMAC signature in a header); check it before trusting the body.
- **Respond fast, then process.** Acknowledge receipt immediately and do the real work afterwards (from a queue, or a folder, as here), because a slow response makes the sender retry, and you'll receive the same event twice. (The sandbox itself breaks this rule when it calls the webhook inside its own request; a real provider doesn't.)
- **Expect duplicates.** Providers document that webhooks are **at-least-once** (Chapter 50's term, exactly): de-duplicate by the event's own id.
- **Handle being unreachable.** If your receiver is down, does the sender retry, and for how long? Know the answer before you rely on it, and keep a scheduled reconciliation sync as a safety net under any webhook.

### Signatures: HMAC

An **HMAC** (hash-based message authentication code) is a hash of the message mixed with a secret that only the sender and the receiver know. The CRM computes it from the body it sends and puts it in a header, `X-Signature`. The receiver computes it again from the body it received. If they match, the body came from someone who knows the secret and wasn't changed on the way; a forger without the secret can't produce the right value.

In [20]:
SECRET = os.environ.get("WEBHOOK_SECRET", "practice-secret-51").encode()
body = b'{"event_id": "evt_100", "event": "test"}'
signature = hmac.new(SECRET, body, hashlib.sha256).hexdigest()
print("genuine :", signature[:16])
forged = b'{"event_id": "evt_100", "event": "test!"}'
print("forged  :", hmac.new(SECRET, forged, hashlib.sha256).hexdigest()[:16])
print("same?   :", hmac.compare_digest(signature, hmac.new(SECRET, body, hashlib.sha256).hexdigest()))

genuine : 4f679cd25f9d90b7
forged  : 1d3d91ecca2d9904
same?   : True


- `b'...'` is a **bytes** value, the raw form a request body travels in. `.encode()` turns the secret's text into bytes too.
- `hmac.new(secret, message, hashlib.sha256)` mixes the two with SHA-256; `.hexdigest()` gives the result as text. The printout shows the first 16 characters.
- One extra character in the body gives a completely different signature.
- **`hmac.compare_digest(a, b)`** compares two signatures. Use it instead of `==`: it takes the same time whether the first character differs or the last, so an attacker can't guess a signature one character at a time by timing the answers.

The receiver in `webhook_receiver.py` does all four things from the list above. Here is its request handler. You don't run this cell; it's inside the receiver, which the notebook started in section 51.0:

In [21]:
    def do_POST(self):
        body = self.rfile.read(int(self.headers.get("Content-Length", 0)))
        expected = hmac.new(SECRET, body, hashlib.sha256).hexdigest()
        if not hmac.compare_digest(expected, self.headers.get("X-Signature", "")):
            return self.reply(401, {"ok": False, "error": "bad signature"})
        event = json.loads(body)
        if event["event_id"] in Handler.seen:
            return self.reply(200, {"ok": True, "duplicate": True})
        Handler.seen.add(event["event_id"])
        save_to_inbox(event)
        return self.reply(200, {"ok": True})

> **Not run here.** The chapter marks this block as illustration rather than something to execute, usually because it needs a service, a key or a file this notebook does not set up. Read it, do not run it.

- `self.rfile.read(...)` reads the raw body, exactly `Content-Length` bytes. The signature is checked on these bytes, **before** anything else, and a mismatch gets **401** with nothing kept.
- `json.loads(body)` turns the checked body into a dictionary. `Handler.seen` is a set of event ids already received; a repeat is answered 200, so the sender stops retrying, and skipped.
- `save_to_inbox()` writes the event to the inbox folder, and the handler answers 200 at once. Any slow work happens later, from the inbox.

Now send it the genuine request, the same request again, and the forged body with the genuine signature:

In [ ]:
signed = {"Content-Type": "application/json", "X-Signature": signature}
for label, data in [("genuine", body), ("same again", body), ("forged", forged)]:
    r = requests.post("http://127.0.0.1:8052/", data=data, headers=signed, timeout=TIMEOUT)
    print(label, r.status_code, r.json())
print("events in the inbox:", len(read_inbox()))

> **Not run here.** This cell depends on a cell above that could not run here, so its names were never created. Run it on your own machine, with the setup the chapter describes, and you should see what the chapter shows.

**What the chapter shows:**

```
genuine 200 {'ok': True}
same again 200 {'ok': True, 'duplicate': True}
forged 401 {'ok': False, 'error': 'bad signature'}
events in the inbox: 2
```

- **`data=`** sends the bytes exactly as they are. (`json=` would turn a dictionary into new bytes, and the signature must match the bytes that were signed.)

**Reading it.** The genuine event was accepted, its repeat was acknowledged as a duplicate and skipped, and the forged body was refused with 401. The inbox holds two events: Grand Horizon Hotels' `evt_1` and the test's `evt_100`.

---

## 51.7 Reconciliation for writes

Chapter 47 reconciled what a pipeline **read**. The same discipline applies to what it **writes**, and it's skipped just as often. Reconciling a write means reading the destination back and comparing it with what the warehouse says it should hold.

First, read everything back, page by page:

In [ ]:
def read_all(kind, id_field, page_size=20):
    records, page, pages = [], 1, 0
    while page is not None:
        resp = requests.get(f"{BASE}/{kind}", params={"page": page, "page_size": page_size},
                            headers=HEADERS, timeout=TIMEOUT)
        resp.raise_for_status()
        body = resp.json()
        records.extend(body["data"])
        page, pages = body["next_page"], pages + 1
    return {r[id_field]: r for r in records}, pages

crm_leads, pages = read_all("leads", "lead_id")
print("leads:", len(crm_leads), "in", pages, "pages")
crm_accounts, pages = read_all("accounts", "customer_id")
print("accounts:", len(crm_accounts), "in", pages, "pages")

> **Not run here.** This cell depends on a cell above that could not run here, so its names were never created. Run it on your own machine, with the setup the chapter describes, and you should see what the chapter shows.

**What the chapter shows:**

```
leads: 44 in 3 pages
accounts: 24 in 2 pages
```

- **`params={"page": page, "page_size": page_size}`** adds the two settings to the address, as `?page=1&page_size=20`. `page_size=20` asks for 20 records a page, so the 44 leads take three pages; with `page_size=50` (the sandbox's largest) they would take one, and with 10, five.
- The `while` loop is Chapter 45's pagination: ask for page 1, then whatever `next_page` the CRM returns, until it's `None` on the last page. `pages` counts the requests.
- The dictionary comprehension indexes the records by their id, so a lead can be looked up as `crm_leads["4"]`. `id_field` names the id column, which differs between leads and accounts.
- The function returns two values, the records and the page count, and `crm_leads, pages = ...` unpacks them.

**Reading it.** 44 leads, the 43 Riverstone leads plus Grand Horizon Hotels, created in the CRM in section 51.6. 24 accounts, including the one the upsert created.

Now compare, record by record:

In [ ]:
def find_mismatches():
    crm_leads, _ = read_all("leads", "lead_id")
    crm_accounts, _ = read_all("accounts", "customer_id")
    found = []
    for lead_id, s in scores.items():
        crm_value = crm_leads[lead_id].get("score")
        if crm_value != s["score"]:
            found.append((f"leads/{lead_id}", s["score"], crm_value))
    for cid, due_flag in follow_up.items():
        crm_value = crm_accounts[cid].get("follow_up_due")
        if crm_value != due_flag:
            found.append((f"accounts/{cid}", due_flag, crm_value))
    return found

print("checked:", len(scores), "leads and", len(follow_up), "accounts")
for record, warehouse_value, crm_value in find_mismatches():
    print(f"MISMATCH {record}: warehouse {warehouse_value}, CRM {crm_value}")

> **Not run here.** This cell depends on a cell above that could not run here, so its names were never created. Run it on your own machine, with the setup the chapter describes, and you should see what the chapter shows.

**What the chapter shows:**

```
checked: 43 leads and 24 accounts
MISMATCH leads/4: warehouse 20, CRM 21
```

- `find_mismatches()` reads the CRM again each time it's called, so it always compares with the CRM as it is now. `_` is the usual name for a value you don't need, here the page count.
- It loops over the **warehouse's** records, not the CRM's, because the question is "did everything we meant to write arrive?". Grand Horizon Hotels isn't in the warehouse yet (tomorrow's ingestion will bring it), so it isn't checked.
- **`.get("score")`** returns `None` instead of stopping with an error if the CRM record has no score, for example a record the sync never reached.

**Reading it.** One mismatch: lead 4, 20 in the warehouse against 21 in the CRM. That's section 51.4's +1, which the content-only key then couldn't undo. The report caught it the same day, which is its job. Re-running today's sync wouldn't fix it: every write would be a replay, as "back to 20" showed.

### Tomorrow's run, with the run-id key

Here is the nightly sync as the project would run it: every lead and every account, keyed with `run_key()`, with each write's result recorded:

In [25]:
def nightly_sync(run_id, run_time):
    jobs = [(f"leads/{i}", {"score": s["score"], "stage": s["stage"]}) for i, s in scores.items()]
    jobs += [(f"accounts/{c}", {"follow_up_due": d}) for c, d in follow_up.items()]
    log = []
    for record, payload in jobs:
        key = run_key(record, payload, run_id)
        body = {**payload, "score_updated_at": run_time} if record.startswith("leads/") else payload
        try:
            result = sync_record(record, body, key)
            outcome = "replayed" if result["replayed"] else "new"
        except (requests.HTTPError, RuntimeError) as e:
            outcome = "rejected: " + str(e)
        log.append({"run": run_id, "record": record, "sent": payload, "key": key, "outcome": outcome})
    with open("sync_state/audit_log.jsonl", "a") as f:
        for row in log:
            f.write(json.dumps(row) + "\n")
    return log

- `jobs` is the list of writes: one `(record, payload)` pair per lead, then `+=` adds one per account.
- Each write gets its `run_key()`. Leads also carry `score_updated_at`; accounts don't. `x if condition else y` picks one of two values (Chapter 17).
- **`try` / `except`** (Chapter 17) catches a write the CRM refused, such as a 404, or one that failed every attempt, and records it as **rejected** instead of stopping the whole run: one bad record shouldn't block the other 66.
- `log` gets one row per write. The `with open(..., "a")` block **appends** every row to `sync_state/audit_log.jsonl`, one JSON object per line, a common format for logs called JSON Lines. That file is the audit trail below.

Now pretend it's 7 January, the warehouse's values haven't changed overnight, and run it, then reconcile:

In [ ]:
log = nightly_sync("2026-01-07", "2026-01-07T06:30:00+05:30")
outcomes = [row["outcome"] for row in log]
print("attempted :", len(log))
print("confirmed :", sum(1 for o in outcomes if not o.startswith("rejected")))
print("new       :", outcomes.count("new"))
print("replayed  :", outcomes.count("replayed"))
print("rejected  :", sum(1 for o in outcomes if o.startswith("rejected")))
print("mismatched:", len(find_mismatches()))
print("audit row :", log[3])

> **Not run here.** This cell depends on a cell above that could not run here, so its names were never created. Run it on your own machine, with the setup the chapter describes, and you should see what the chapter shows.

**What the chapter shows:**

```
attempted : 67
confirmed : 67
new       : 67
replayed  : 0
rejected  : 0
mismatched: 0
audit row : {'run': '2026-01-07', 'record': 'leads/4', 'sent': {'score': 20, 'stage': 'New'}, 'key': '67329d74f025d291', 'outcome': 'new'}
```

- `sum(1 for o in outcomes if ...)` counts the items that pass the test: it adds 1 for each. `outcomes.count("new")` counts exact matches.
- `log[3]` is the fourth write, lead 4's, as it's stored in the audit trail.

**Reading it.** All 67 writes, 43 leads and 24 accounts, were attempted and confirmed, none rejected, and the reconciliation now finds **no mismatches**: with a new run id, lead 4's 20 was applied again. Every write was new, though, even the 66 whose values hadn't changed. That's the price of the run-id key: each run re-sends everything. At thousands of records you'd send only the rows whose values changed since the last run, Chapter 45's incremental idea (section 45.4) applied to writes, and keep the run id in the key for those.

**A reverse-ETL reconciliation report should show, every run:**

- how many records the warehouse tried to sync;
- how many the destination confirmed;
- how many writes were replays (already applied) versus new changes;
- anything the destination rejected, with its error;
- any record where the destination's value doesn't match what was sent, with both values.

This is Chapter 46's gating check, pointed the other way: **hold the "sync succeeded" status, and alert, until the reconciliation confirms it**, exactly as the Daily Sales Flash isn't delivered until its numbers match the ERP.

### An audit trail

Keep a durable log of every write attempt, separate from the CRM's own record: time, record, old value if known, new value, idempotency key, and outcome. `nightly_sync()` keeps a simple one in `audit_log.jsonl`; a production version also stores the time of each write and the CRM's old value, read just before writing. When someone asks "why does this lead have this score", or "who changed this field last Tuesday", the audit trail answers it without guessing. It's cheap to build (one row per write), and it's the first thing anyone reaches for during an incident.

---

## 51.8 Integration patterns

As the number of systems grows, *how* they connect matters as much as any single sync.

![Four small diagrams side by side. Point-to-point shows five systems with a separate line between every pair, ten lines in all, labelled up to n(n−1)/2 links, 5 systems give 10. Hub-and-spoke shows the same five systems each connected only to a central hub, labelled n links, 5 systems give 5. Message queue shows systems connected by dashed lines to a queue in the middle, labelled senders and receivers don't know each other. iPaaS shows a box labelled integration platform where the hub sat, with the note: a hosted hub, with connectors, monitoring and retries included.](figures/fig51-2-integration-patterns.svg)

*Figure 51.2 — Four ways to wire systems together. The number of connections is the thing to watch as the company grows.*

| Pattern | Shape | Strength | Weakness |
|---|---|---|---|
| **Point-to-point** | Every system talks directly to every other | Simple to start, no new infrastructure | Grows roughly as the square of the system count: n systems need up to n(n−1)/2 links (5 systems, 10 links; 10 systems, 45); nobody can see the whole picture |
| **Hub-and-spoke** | Every system talks only to a central hub | New systems add one connection, not many | The hub becomes critical; needs its own reliability |
| **Message queue / event bus** | Systems publish and subscribe to events, decoupled | Producers and consumers don't need to know about each other; naturally handles bursts | Needs the infrastructure from Chapter 50; harder to trace a single request end to end |
| **iPaaS** (integration platform as a service) | A hosted hub with pre-built connectors, retries, and monitoring | Fast to build with, less code to own | Ongoing cost; you depend on the vendor's connector for each system |

Riverstone at its current size (a handful of systems: ERP, CRM, the warehouse, one or two marketing tools) is well served by **hub-and-spoke with the warehouse as the hub**, which is exactly what Chapters 45–51 have built without naming it: everything reads from and writes through the warehouse and its pipelines, rather than the CRM talking directly to the ERP. Message queues and iPaaS earn their cost once there are many systems, many events, or a team too small to maintain custom connectors for each one.

---
## 51.9 Choosing how to build it

| Option | Best for | Cost |
|---|---|---|
| **Custom code** (this chapter) | A handful of well-understood syncs; full control over idempotency and reconciliation | Engineering time to build and maintain |
| **Orchestrator tasks** (Chapter 46) | Syncs that fit naturally next to your existing pipelines | Some setup; scales with what you already run |
| **Low-code / iPaaS** (Zapier, Workato, Tray.io, n8n) | Many simple syncs, built quickly, by people who aren't primarily engineers | Per-task or per-connector pricing; less control over retries and idempotency |
| **Enterprise integration platforms** (MuleSoft, Boomi, Informatica) | Large organizations with many systems and compliance needs | Significant cost and specialist skill to run |
| **Vendor-native reverse ETL** (Census, Hightouch) | Syncing warehouse tables into standard SaaS tools with minimal code | Subscription cost; excellent fit when the destination is a well-supported SaaS app |

The decision tree is short: **if a reverse-ETL tool already supports your exact destination and the sync is "a warehouse column into a CRM field," use it**, since idempotency and pagination are solved for you. **Build custom code when the logic is bespoke** (Riverstone's lead score, a conflict rule specific to your business) or when no connector exists for your system, as is common with older ERPs.

---

## 51.10 When there's no API: RPA and file drops

Not every system Riverstone deals with has an API. Some suppliers' portals only offer a login page. Some of Riverstone's own older tools were never built to be integrated with.

### RPA, and why it's fragile

Chapter 7 introduced **RPA** (robotic process automation): software that clicks through a screen the way a person would, reading and typing into the same interface a human uses. It's sometimes the only option, and it's genuinely useful when there's truly no other way in.

It's fragile because it depends on things that were never a contract: a button's position, a label's exact wording, how long a page takes to load. A vendor redesigns their portal, and every recorded click breaks at once, silently, often producing wrong data rather than an obvious error, because the bot clicked whatever is now in that position on the screen.

**If you must use RPA:** treat every run as production-critical (monitoring, alerts, a human review step for anything unusual), prefer tools that read the page's underlying structure over raw pixel coordinates, and revisit the decision every time the target system changes. An API sometimes appears later, or a data export replaces the need entirely.

### File-based integration: still normal, not a failure

**SFTP drops** (a file placed on a secure server on a schedule) and **EDI** (Electronic Data Interchange, a decades-old standard format for business documents like purchase orders and invoices) remain the backbone of a huge amount of manufacturing and B2B integration. A supplier who has exchanged EDI purchase orders with hundreds of partners for twenty years has no reason to build you an API.

Everything Chapter 45 taught about files applies directly: declared columns, a load log, reconciliation against a control total, and a data contract with the sending party about format and timing. The only difference here is direction: you may be the one **producing** the file, in which case the same discipline applies to what you send, with a stable format, a clear naming convention, and validation before it leaves, so a malformed export doesn't become someone else's 3 a.m. incident.

---

## Common mistakes

| Mistake | Symptom | Fix |
|---|---|---|
| Building a beautiful dashboard nobody visits | The insight exists; nothing changes | Push it where the decision is already made (51.1) |
| Syncing a field without checking who else edits it | A human's careful edit vanishes on the next sync | Name the system of record per field (51.5) |
| No idempotency key on writes | A retried sync doubles a discount or creates a duplicate lead | Send a key with every write, and prove replays are no-ops (51.3, 51.4) |
| Key built inside the retry loop, from "now" or a fresh random value | The key changes every retry, so it protects nothing | Create the key once, before the first attempt, and reuse it on every retry (51.4) |
| Key built from the values alone | A value that goes back to an earlier one is never sent again; reconciliation flags the same record every night | Add the run id to the key (51.4) |
| Retrying only one kind of failure | A timeout or a 429 crashes the sync halfway | Retry 429, 503, timeouts, and dropped connections, respecting `Retry-After`; fail fast on other errors (51.3) |
| Using `PUT` to change one field | Every field you didn't send is wiped | `PATCH`, naming only the fields the sync owns (51.3) |
| Trusting a write "succeeded" because no exception was thrown | A field silently didn't take, and nobody notices for weeks | Reconcile writes the same run, not later (51.7) |
| Polling every few minutes when the moment matters | A high-value lead sits unnoticed for most of the interval | A webhook, with a reconciliation sync as a backstop (51.6) |
| Treating a webhook receiver as always-on and always-fast | Duplicate events; missed events during downtime | Verify signatures, respond fast, de-duplicate, expect retries (51.6) |
| Point-to-point connections between every pair of systems | Adding one system means touching five others | Hub-and-spoke, or a queue, as the system count grows (51.8) |
| Choosing RPA when an API exists | Fragile, breaks on every UI change | Check for an API or export first; RPA is a last resort (51.10) |
| No audit trail for writes | "Why does this record say this?" has no answer | Log every write attempt with old value, new value, and outcome (51.7) |
| Building custom code for a well-supported SaaS destination | Reinventing pagination, retries, and auth a tool already solved | Check for a reverse-ETL or iPaaS connector first (51.9) |

---

## In the real world: the sync that erased a discount

Three months after Riverstone's lead-scoring sync went live, a sales rep phoned Meera, upset. A customer she'd spent weeks negotiating a loyalty discount for had opened their CRM record that morning to find the discount field blank.

Meera traced it in under an hour, because the audit trail this chapter builds made it possible. The nightly reverse-ETL job wasn't touching discounts: it only wrote lead scores. But a second sync, built six weeks earlier by a different contractor to push "customer segment" into the CRM, had been written against an older export of the customer table that didn't include the newly added discount column. Its update statement replaced the **entire customer record** rather than patching one field, because the contractor had used `PUT` (replace everything) instead of `PATCH` (change only what's specified), and nobody had reconciled the write afterward.

The discount wasn't malicious, and it wasn't even the sync this chapter is about. It was a `PUT` where a `PATCH` was needed, on a field nobody had named a system of record for, with no reconciliation to catch it, and no audit trail on that particular job because it had been built in a hurry outside the team's normal process.

The fixes were exactly this chapter's checklist. Every write became a `PATCH` naming its fields explicitly, never a full-record replace. Every synced field got an entry in a small internal document naming its system of record: discount is **the CRM**, never written by any pipeline. Every sync gained the same reconciliation report as the lead-score job. And the "customer segment" sync, rebuilt properly, took half a day.

At the retrospective, the plant manager's line from Chapter 50 came up again, adapted: *a sync that silently overwrites what a person entered is not a small bug. It's a system nobody can trust with anything a person has typed by hand.*

**What made this work.**

- **The cause wasn't the sync this chapter builds**: it was a different one, built without the same discipline, which is exactly why the discipline has to be a rule for every sync, not a one-off best effort.
- **`PATCH`, not `PUT`**, for anything that shouldn't touch fields it doesn't know about.
- **A named system of record per field** would have made "never write to discount" an explicit, checked rule instead of an assumption.
- **Reconciliation would have caught it the same night**, not three months later from an upset customer.

---

## Project: sync Riverstone's lead scores and follow-up flags

**Goal:** a nightly, idempotent sync of lead scores and follow-up-due flags into the sandbox CRM, with a reconciliation report, plus a webhook that alerts sales within a minute of a high-value lead.

### Tools you'll need

- **Python** with `psycopg2` and `requests`, in the book's virtual environment (installed in Chapters 18 and 45).
- **The Chapter 51 companion folder** (`companion/ch51/`, copied to `work/ch51` in section 51.0): `reset_ch51.py`, `crm_sandbox.py` (the sandbox CRM's API), `webhook_receiver.py` (the local webhook receiver), `lead_scores.sql` and `follow_up_due.sql` (the two queries), and `sync_leads.py` (the scoring, the flags, and both keys, as functions). Run Python from that folder.
- **Versions used for the outputs shown:** Python 3.11.15, requests 2.33.1, psycopg2 2.9.13, PostgreSQL 16.13.
- **Worth knowing about:** reverse-ETL tools (Census, Hightouch), iPaaS platforms (Zapier, Workato, Tray.io, n8n, MuleSoft, Boomi), and RPA tools (UiPath, Automation Anywhere, Power Automate Desktop).

**Option A: Riverstone.** Use the companion environment.

**Option B: your own systems.** Any write API you have permission to use, ideally with a sandbox or test mode. Never practice writes against a production system without explicit permission.

**Steps**

1. **Name the system of record** for every field you'll sync, and write down the conflict rule for each, before writing any code.
2. **Compute the values** to sync (adapt `sync_leads.py`'s approach, or use your own logic), documented in one sentence per rule.
3. **Build the sync function** with an idempotency key built from the record, the values, and the run id, created before the first attempt; retries on 429, 503, timeouts, and dropped connections only; and `PATCH` requests that touch only the fields you intend.
4. **Prove idempotency:** send the same write twice in one run and show the second call is a no-op; change the content and show it's applied; send an earlier value again in a new run and show it's applied too.
5. **Add the reconciliation report**: records attempted, confirmed, mismatched (with both values), replayed versus new, and rejected.
6. **Add an audit trail** table or log: timestamp, entity, fields, idempotency key, outcome.
7. **Register a webhook** for one high-value event, and measure the time from the event to your receiver getting it.
8. **Make the webhook receiver safe**: acknowledge fast, de-duplicate by event id, and add a scheduled reconciliation sync as a backstop in case an event is ever missed.
9. **Write the incident scenario**: what would happen if this sync ran with a stale or wrong value for a day, who would notice, and how they'd find out. Add one check that would have caught it.

**Stretch goals**

- Turn the sync into a Chapter 46 Dagster asset, gated by a Chapter 47 check before it's allowed to write.
- Simulate the CRM being unreachable for a whole run, and design the retry-and-alert behavior (Chapter 46's patterns, applied to writes).
- Compare your custom sync against what a reverse-ETL tool's free tier would let you build for the same fields, and note what you'd trade.

---

## Recap

- **Data activation** closes the loop from source systems through the warehouse and back into the systems people actually work in. A dashboard is pulled; a synced field is pushed to where the decision already happens.
- **Reverse ETL** writes computed fields (lead scores, follow-up flags, health scores, credit holds) into the CRM, ERP, and marketing tools.
- Writes use the same **API mechanics** as reads (authentication, pagination, retries on 429, 503, and timeouts) with sharper consequences, and OAuth is standard for write access. An **upsert** creates a record or updates it, matched on an id both systems share.
- An **idempotency key** makes retries safe: a repeated write returns `replayed: True` and changes nothing. Create it once, before the first attempt, from the record, the values, and the **run id**: a key from the values alone can never re-send a value that went back to an earlier one.
- **`PATCH` changes only named fields; `PUT` replaces the whole record.** Using `PUT` where `PATCH` was needed is a real, common cause of silently erased data.
- Name the **system of record** for every synced field individually, and write the **conflict rule** before two systems disagree in front of a customer.
- **Webhooks** push events the moment they happen; treat receivers as needing signature verification (an **HMAC** of the body with a shared secret), fast acknowledgement, and de-duplication by event id, and always keep a reconciliation sync as a backstop.
- **Reconcile writes**, not only reads, every run, and keep an **audit trail** of every attempt.
- **Integration patterns** (point-to-point, hub-and-spoke, message queues, iPaaS) trade simplicity against how well they scale as systems multiply.
- Choose **custom code, orchestrator tasks, low-code tools, or enterprise platforms** based on how bespoke the logic is and how well-supported the destination already is.
- **RPA** is fragile because it depends on a screen's layout rather than a contract; **file-based integration** (SFTP, EDI) is not a failure mode, it's still normal, and deserves the same discipline as any other file load.

---

## Key terms

data activation · reverse ETL · lead score · customer health score · credit hold · reorder flag · churn risk · follow-up-due flag · scope · service account · OAuth 2.0 · access token · `GET`, `POST`, `PUT`, `PATCH` · path parameter · upsert (by external id) · idempotency key · replay · run id · system of record · conflict rule · webhook · HMAC (event signature) · at-least-once webhook delivery · point-to-point integration · hub-and-spoke integration · message queue / event bus · iPaaS (integration platform as a service) · enterprise integration platform · RPA (robotic process automation) · SFTP · EDI (Electronic Data Interchange) · audit trail · JSON Lines · write reconciliation

*(All terms are defined in the Glossary, Appendix A.)*

---

## Check yourself

- [ ] You can explain why a correct number in a warehouse can still change nothing.
- [ ] You can name three fields worth syncing back into operational systems and what each changes.
- [ ] You can write a `PATCH` request with authentication, and explain why `PATCH` and `PUT` aren't interchangeable.
- [ ] You can build an idempotency key that makes a retry a no-op but lets a new run re-send an earlier value, and prove both.
- [ ] You can explain system of record and write a conflict rule for a specific field.
- [ ] You can register and receive a webhook, check its HMAC signature, and say what else makes a receiver safe.
- [ ] You can reconcile a set of writes and explain what a mismatch means.
- [ ] You can compare point-to-point, hub-and-spoke, message queues, and iPaaS, and say which fits a given company size.
- [ ] You can explain why RPA is fragile and when it's still the right tool.
- [ ] You can describe SFTP and EDI integration and apply Chapter 45's file discipline to a file you produce.
- [ ] You can design an audit trail that answers "why does this record say this?"

---

## Exercises

### Warm-up

1. For each field, say whether the warehouse or a human system should be its system of record, and why: (a) a computed customer health score; (b) a manually negotiated discount; (c) an order's shipping status; (d) a lead score computed nightly.
2. Explain the difference between `PATCH` and `PUT` in one sentence each, and say which one the sync that erased a discount should have used.
3. A webhook fires twice for the same event, five seconds apart. Name two ways your receiver could be built to make that harmless.

### Core

4. Using the rules in section 51.2, compute the lead score for a lead that is **Contacted**, sourced from a **Trade fair**, that entered its stage 3 days before the chapter's "today". Show your working.
5. In section 51.4, lead 4's "same values again" and its "back to 20" both returned `replayed: True`. Explain precisely why, referring to what the key was built from, and say why the first is correct and the second is a bug.
6. Write the idempotency key logic for a sync that should apply "once per scheduled run" rather than "once per distinct value" (the third row of the table in section 51.4). What business situation would make you prefer that behavior?
7. The reconciliation in section 51.7 found one mismatch, for lead 4. Explain what caused it, why re-running the section 51.3 sync couldn't repair it, and what did. Is the mismatch a bug, or the reconciliation working as intended? Justify your answer.
8. Riverstone wants to add a sync for "customer segment" into the CRM. Using section 51.5's framework, write the two-line conflict policy for that field, including what happens if a rep manually changes the segment in the CRM.
9. A supplier sends purchase order confirmations by SFTP once a day, as a fixed-width text file. List the four things from Chapter 45's file discipline you'd apply before trusting a single row of it.

### Stretch

10. Design the reconciliation report you'd run every night for the lead-score sync at real volume (thousands of leads, not 43), including what you'd summarize versus what you'd list row by row, and what would trigger an alert versus a note in a log.
11. A vendor discontinues the CRM's API in favor of a new one with different field names and a different authentication scheme. Walk through what in this chapter's design changes and what stays the same (idempotency keys, reconciliation, system of record, the webhook contract).
12. Compare building the lead-score sync as (a) custom Python in an orchestrator, (b) a low-code iPaaS flow, and (c) a vendor reverse-ETL tool, for a company with one part-time data person. Recommend one, with reasons, and say what would change your recommendation if the company had a five-person data team instead.

### Think about it (no code needed)

13. "A sync that silently overwrites what a person entered is not a small bug." Explain what makes this failure mode worse than a wrong number in a report, using the ideas from Chapter 47 on severity.
14. RPA and file-based integration are both presented in this chapter as reasonable choices in the right circumstances, not as mistakes. Explain to a colleague who wants to "modernize everything to APIs" why that isn't always the right goal.

---

## Answers

**1.** (a) **Warehouse**: it's computed from data the CRM doesn't hold, and no person should hand-edit a calculated score. (b) **The CRM (a human system)**: a person negotiated it, and a pipeline must never silently overwrite that. (c) **The ERP**: it's where the transaction and the physical dispatch happen. (d) **Warehouse**: same reasoning as (a), nightly computed, not entered by a person.

**2.** **`PATCH`** changes only the fields named in the request, leaving everything else untouched. **`PUT`** replaces the entire record with what's sent, so any field left out is wiped. The discount-erasing sync should have used **`PATCH`**, naming only the customer-segment field, so the discount field would never have been touched.

**3.** (1) **De-duplicate by the event's own id**: keep a short-lived record of ids already processed, and skip a repeat. (2) **Make the processing itself idempotent**, in the style of this chapter's writes, so that even without explicit de-duplication, handling the same event twice produces the same end state rather than a doubled effect.

**4.** From section 51.2's tables: Contacted = 30 points (stage), Trade fair = 20 points (source). The stage was entered 3 days before today, inside the 14-day window, and the lead isn't Lost, so +10 recency. 30 + 20 + 10 = **60**, below the cap of 100, so no capping applies.

**5.** Both keys were built from the record and the values only (section 51.3's `idempotency_key`). "Same values again" sent lead 4's score and stage exactly as the first sync had, so the key matched the one the CRM had stored, and it returned its earlier answer, `replayed: True`, without touching the record. That is correct: nothing needed to change. "Back to 20" also sent those same values, after the score had been changed to 21, so it produced the same old key and was ignored too. That is the bug: the CRM still says 21, and no later sync with a content-only key can ever set it back to 20. Adding the run id to the key (section 51.4's `run_key`) fixes it.

**6.** A sample answer, the third row of the table in section 51.4, built like `run_key()` without the values:

In [27]:
def run_scoped_key(record, run_id):
    content = record + ":" + run_id
    return hashlib.sha256(content.encode()).hexdigest()[:16]

> **Not run here.** The chapter marks this block as illustration rather than something to execute, usually because it needs a service, a key or a file this notebook does not set up. Read it, do not run it.

- `record` is the record's address, such as `"leads/4"`, and `run_id` is the run's day, such as `"2026-01-07"`. The payload is left out on purpose.
- `content.encode()` turns the text into bytes, `hashlib.sha256(...).hexdigest()` fingerprints it, and `[:16]` keeps 16 characters, exactly as in `idempotency_key()`.
- `return` hands the key back to the caller, which creates it once, before the first attempt.

This makes retries within the *same scheduled run* a no-op regardless of whether the score changed mid-run, but a *new run* always gets a new key, so its write is applied even if the value happens to be identical to last time. You'd prefer this when you want a clean, auditable "one write per run" record, for example a compliance requirement that every nightly sync is logged exactly once per lead per night, rather than being silently skipped because the value hadn't moved.

**7.** Section 51.4 pushed a "+1" test value into lead 4 (21), standing in for a genuine change, and then tried to send the warehouse's 20 again. With a key built from the values alone, that write had the same key as the first sync's, so the CRM replayed it and kept 21. Re-running the section 51.3 sync would produce exactly the same keys, so every write would be replayed and lead 4 would stay wrong forever. What repaired it was the next run with `run_key()`: a new run id gives new keys, so 20 was applied, and the next reconciliation was clean. The mismatch itself was the reconciliation **working as intended**: it caught the drift the same day. The bug was the key design, which is what to fix. In a real deployment the same report line needs investigating: was it a manual edit in the CRM (perhaps this field shouldn't be blindly overwritten), a partly failed sync, or a second job writing the same field (as in the closing story)? The report's job is to surface the question, not answer it.

**8.** A sample two-line policy: *"Customer segment: system of record is the warehouse; the nightly sync always overwrites this field with the computed value. If a rep manually changes it in the CRM, that change will be overwritten by the next sync within 24 hours; reps should request a change to the segmentation rule instead of hand-editing individual records."* The key elements: name the system of record explicitly, and say plainly what happens to a manual edit, so nobody is surprised later.

**9.** (1) **Declare the columns and their types explicitly** rather than letting a reader guess from a fixed-width file with no header. (2) **Land the raw file unchanged** before parsing, so the original is always available if the parsing rule turns out wrong. (3) **Reconcile row counts (and a control total if the supplier provides one) against the file** before trusting the loaded data. (4) **Check for schema drift**: confirm the field widths and layout match what was agreed, and stop with a clear error if a column has shifted, rather than silently misreading every field after it.

**10.** At real volume: **summarize** by default, with total attempted, total confirmed, total replayed, total newly changed, total mismatched, total rejected, broken down by error type for rejections, and only **list row by row** the records that are mismatched or rejected, since those are what a person needs to act on; a report listing every one of ten thousand successful matches is noise. **Alert** (page someone) when: the mismatch or rejection rate crosses a threshold (say, more than 1% of attempted writes), when the whole sync fails to run, or when a previously-matching record now mismatches for no logged reason. **Log without alerting**: normal-range mismatches under investigation, expected rejections (like a lead deleted in the CRM since the last run), and routine replay counts.

**11.** **Changes:** the authentication code (new OAuth flow instead of a bearer token), the field names in the request and response bodies, and the base URL and any pagination parameter names. **Stays the same:** the idempotency key design (still built from your own entity id and content, not from anything the new API dictates), the reconciliation report's logic (still comparing your computed value against what the destination confirms), the system-of-record decisions (unchanged by which vendor holds the CRM), and the webhook contract's shape, if the new API also supports webhooks (register a URL, verify signatures, de-duplicate by event id) even though the exact payload will differ. The lesson: a well-designed sync isolates vendor-specific details (auth, field names, URLs) behind a thin layer, so a vendor migration touches a small, obvious part of the code.

**12.** For **one part-time data person**: recommend **(c) a vendor reverse-ETL tool** if the CRM is a well-supported mainstream product, because idempotency, pagination, retries, and authentication are solved by the vendor, leaving the person to define the one thing that's actually specific to Riverstone: the lead-scoring SQL. Custom code (a) demands ongoing maintenance a part-time person can't reliably provide; a low-code iPaaS flow (b) is a reasonable middle ground if no reverse-ETL connector exists for the destination. With a **five-person data team**, the calculus shifts toward **(a) custom code in the orchestrator**, because the team can build it once with full control over reconciliation, idempotency keys tailored to the business's conflict rules, and audit trails that match their own standards. At that scale, avoiding per-row vendor pricing on a reverse-ETL tool also starts to matter, and the team has the capacity to own what they build.

**13.** A wrong number in a report is a **single, visible, correctable** failure: someone reads it, it's wrong, it gets fixed, and the damage is bounded to whoever acted on that one report before the correction went out (both failures are S1 under Chapter 47: wrong data has reached people; the difference is the blast radius and how visible it is). A sync that silently overwrites a human's entry is worse on every axis: it's **invisible** until someone happens to look at the specific record; it **destroys information** that may not exist anywhere else (the negotiated discount lived only in that field); it can **recur indefinitely**, silently re-erasing a fix every time the sync runs, until someone identifies the actual cause; and it **damages trust in the system itself**, since once a rep learns the CRM can silently lose their work, they stop trusting the sync for everything, not just the one field that broke.

**14.** The goal was never "have an API". It was **reliable, correct data movement between systems, at a cost proportional to what's at stake.** An API is usually the best way to get that, but a twenty-year-old EDI relationship with a supplier that has worked reliably for two decades, validated by the same file-checking discipline as any other data source, isn't broken just because it's old; rebuilding it around a new API the supplier doesn't have would cost real effort to deliver the same reliability that already exists. Similarly, RPA against a portal with no API isn't a failure of engineering: it's occasionally the only bridge available, and the right response is to operate it carefully (monitoring, human review, revisiting the decision as the target changes), not to pretend it doesn't exist or refuse to use it. "Modernize everything" is a good instinct pointed at the wrong target: point it at reliability and cost, and let the technology choice follow from that, case by case.

---

## Where this leads

- **Chapter 52, The Cloud, Containers & Infrastructure as Code,** deploys syncs like this one somewhere they stay running: secrets management for tokens such as `CRM_API_TOKEN` (section 52.6), containers, and scheduled infrastructure.
- **Chapter 47** applies directly: the reconciliation and audit-trail habits here are Chapter 47's checks, pointed at writes instead of reads.
- **Chapter 46** is where a real deployment of this sync would live, as an orchestrated, idempotent, checked pipeline step.
- **Chapter 50**'s at-least-once delivery is the same guarantee webhooks make, from the other direction.
- **Chapter 58, Intelligent Automation,** builds AI-driven actions on top of exactly this activation layer, with the same idempotency and system-of-record discipline.
- **Chapter 63, Automation Architecture & Governance,** returns to integration patterns at the whole-company level.
- **Part 8:** Chapter 78 (Automation & Integration) drills webhooks, idempotency, and CRM syncs; Chapter 77 includes system design cases built on exactly this loop.